# Mini Bookstore — Learn Tool Calling Step by Step

### What you will learn

By the end of this activity, you should be able to:

1. Explain what a Python function is and why an LLM might need to call one.
2. Distinguish a **tool definition**, a **model-requested tool call**, and **actual Python execution**.
3. Describe how LangChain and Pydantic provide tool descriptions and argument schemas.
4. Follow an agent through multiple model turns and returned tool results.
5. Compare local Qwen with a model hosted by Groq using the same tools and data.
6. Check that an answer is supported by verified tool results.

### Preparation

- Open this notebook in **Google Colab** and select **Runtime → Change runtime type → GPU**. A GPU is required for the local Qwen example.
- Run notebook cells **from top to bottom**. The model download can be substantial, particularly the first time.
- You will use four books stored directly in Python; no database is required in this short activity.
- To use the hosted Groq model, create an account at [Groq Console](https://console.groq.com/), create an API key at [API keys](https://console.groq.com/keys), and save it in **Colab → Secrets** as `GROQ_API_KEY`. Enable notebook access. **Never put a secret key directly in your notebook.** Groq usage is subject to [rate limits](https://console.groq.com/docs/rate-limits).

### How to use the code cells

- `%%writefile filename.py` writes the cell's source code into a real Python file in Colab.
- `!python filename.py` runs that file in a **separate Python process**. This is useful for setup and simple tests.
- `%run -i filename.py` executes a file using the **current notebook's variables**, allowing us to keep the loaded model and traces available for later cells.

**Warnings versus errors:** An optional Qwen acceleration kernel warning may only mean slower inference. An `ImportError` or a failing cell means the environment needs attention. The installation uses **unrestricted `pillow`** instead of forcing a version; Pillow is imported by parts of Transformers even though this demo does not use Gradio. If you changed Pillow versions in a running session and encounter `ImportError: cannot import name '_Ink' from 'PIL._typing'`, restart the Colab runtime and rerun setup in order. A fresh runtime is often the cleanest solution.

This short demo prepares you to investigate the longer **Bookstore Lab (Parts A–I)** independently.

## Step 1 — Start with ordinary Python functions

An LLM cannot automatically read your Python variables or execute your code. First, we will build an extremely small bookstore **without a model or an agent**. The data is fictional teaching data, not a live online bookshop.

We keep the two kinds of information separate:

| Catalog (`BOOKS`) | Ratings (`RATINGS`) |
|---|---|
| Book ID, title, genre, price | Book ID, average rating, review count |

Our two functions will be:

- `search_books(genre, max_price)`: return books matching a genre and maximum price; **ratings are not included**.
- `get_book_rating(book_id)`: return the rating of one book identified by its numeric ID.

The distinction matters for a question such as: **“Find a sci-fi book under $16 with a rating above 4.5.”** One function identifies a candidate; the other verifies its rating. Think about which function you would call first.

### Read the Python code before running it

The next cell creates `demo_bookstore.py`. Notice the following parts:

- `BOOKS` is a **list** of dictionaries (`{...}`). Each dictionary represents one book.
- `RATINGS` is a **dictionary** indexed by the book's numeric ID. Looking up `RATINGS[1]` gives Dune’s rating.
- `def search_books(...)` defines a **function** with two inputs. It filters the catalog using a list comprehension.
- `def get_book_rating(...)` defines another function. It checks whether a book ID exists and returns a structured result.
- The `raise ValueError(...)` lines reject unsuitable arguments. Correct types matter when LLMs propose function parameters.
- The block under `if __name__ == "__main__":` runs only when this file is executed directly, not when another file imports its functions.

`list[dict]` and `-> dict` are **type hints**: they communicate intended input/output types to readers and developer tools. Python still needs runtime checks for untrusted inputs.

You do **not** need to memorize every Python expression. Focus on each function's **inputs, output, and responsibility**.

In [ ]:
%%writefile demo_bookstore.py
"""Tiny bookstore: ordinary Python. No LLM, LangChain, API or database."""

# A list of small records: each dictionary is one book.
BOOKS = [
    {"id": 1, "title": "Dune", "genre": "sci-fi", "price": 14.99},
    {"id": 2, "title": "Foundation", "genre": "sci-fi", "price": 18.50},
    {"id": 3, "title": "The Hobbit", "genre": "fantasy", "price": 12.99},
    {"id": 4, "title": "The Glass Room", "genre": "mystery", "price": 13.50},
]

# Ratings are deliberately separate: the catalog search cannot answer
# a rating question without calling get_book_rating.
# Ratings are stored separately from the searchable catalog.
RATINGS = {
    1: {"average_rating": 4.67, "review_count": 3},
    2: {"average_rating": 4.55, "review_count": 5},
    3: {"average_rating": 4.80, "review_count": 8},
    4: {"average_rating": 4.20, "review_count": 4},
}


def search_books(genre: str, max_price: float) -> list[dict]:
    """Find books of a genre costing no more than max_price, with no ratings."""
    if not isinstance(genre, str) or not genre.strip():
        raise ValueError("genre must be nonempty text")
    if isinstance(max_price, bool) or not isinstance(max_price, (int, float)) or max_price < 0:
        raise ValueError("max_price must be a nonnegative number")
# A list comprehension filters the catalog and makes copies of matching records.
    return [book.copy() for book in BOOKS
            if book["genre"] == genre.strip().casefold() and book["price"] <= max_price]


def get_book_rating(book_id: int) -> dict:
    """Retrieve the rating of a known catalog ID, or say it was not found."""
    if type(book_id) is not int or book_id <= 0:
        raise ValueError("book_id must be a positive integer")
    if not any(b["id"] == book_id for b in BOOKS):
        return {"book_id": book_id, "found": False}
    return {"book_id": book_id, "found": True, **RATINGS[book_id]}


# Tests only run when we execute this file directly, not when importing it.
if __name__ == '__main__':
    print("Catalog:")
    for book in BOOKS:
        print(" ", book)
    print("\nTest 1 — search_books('sci-fi', 16):")
    result = search_books("sci-fi", 16)
    print(result)
    assert [b["id"] for b in result] == [1]
    print("\nTest 2 — get_book_rating(1):")
    rating = get_book_rating(1)
    print(rating)
    assert rating["average_rating"] == 4.67
    print("\nTests passed: no model or network required.")

### A closer look at the return values

`search_books` returns a **list**, because a search can match zero, one or several books. `get_book_rating` returns a **dictionary**, because we request one specific book ID.

The code uses `book.copy()` so callers receive copies of book dictionaries rather than modifying the original catalog by accident. It also checks that the price limit is nonnegative and that the book ID is a positive integer.

**Try reasoning about the code:** If you call `search_books("sci-fi", 20)`, would *Foundation* now appear? If you call `get_book_rating(999)`, what does the `found` field tell you?

### Run the two functions without an LLM

The previous cell **saved** the code to a file; it did not run the tests. This cell executes the file as Python. Look for:

1. A printed catalog with four books.
2. The result of `search_books("sci-fi", 16)` — only Dune should be returned.
3. The rating returned by `get_book_rating(1)` — `4.67` based on three reviews.
4. The message confirming that the assertions passed.

An `assert` is a simple check: execution stops if its condition is false.

In [ ]:
!python demo_bookstore.py

### Checkpoint 1 — Why two functions?

Explain, in your own words:

**Why can `search_books` find Dune but cannot decide on its own whether Dune has a rating above 4.5?**

Consider a second scenario: What should happen if you search for a fantasy book under `$10` and no books match? The appropriate result is an **empty list**, not an invented book.

## Step 2 — Make the functions available as tools

A **tool** is a Python function packaged with information that lets a model know **when and how it may request that function**.

We will use **LangChain Core** to give each function a public name, a description and an **argument schema**. A schema describes acceptable inputs, such as `genre` (text), `max_price` (number) or `book_id` (positive integer).

**Important:** Registering a tool does not make a model execute Python. Our application is still responsible for validating a request and running the actual function.

Before wrapping the functions, we'll install the needed libraries.

### The installation script

`demo_install.py` installs:

| Package | Role in the lab |
|---|---|
| `transformers` | Loads Qwen and formats model messages |
| `accelerate` | Helps place the model on the GPU |
| `bitsandbytes` | Supports 4-bit weight quantization to reduce memory needs |
| `langchain-core` | Provides the `@tool` decorator and tool interfaces |
| `pydantic` | Defines and validates tool argument schemas |
| `groq` | Calls the hosted model API |
| `pillow` | Supports image handling imported by model-processing libraries |



In [ ]:
%%writefile demo_install.py
"""Install dependencies once in Colab (with a GPU runtime selected)."""
import subprocess, sys
# No Pillow version is pinned; Colab manages a shared environment.
packages = ["transformers>=4.57.0", "accelerate", "bitsandbytes",
            "langchain-core>=1.0,<2", "pydantic>=2,<3", "groq", "pillow"]
print("Installing:", ", ".join(packages), flush=True)
# Install into the same Python interpreter used to run this script.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])
print("Dependencies installed. If Transformers was imported before installation, restart the runtime.")

### Execute the installer

This call runs `demo_install.py` as an independent process. Installing changes the Colab environment, so the model does not need to be loaded into notebook memory at this stage.

**Important environment note:** If you change installed packages after importing them in the same runtime, Colab may continue using older imported modules. If you experience an import mismatch, use **Runtime → Restart session**, then rerun the notebook in order. In particular, `_Ink` missing from `PIL._typing` is an actual import error, not an optional-kernel warning.

In [ ]:
!python demo_install.py

### Read the tool wrapper code

`demo_toolkit.py` does several jobs. It is longer than the first file because it connects **ordinary Python** to a **language-model-facing interface**.

**A. Describe input formats with Pydantic.** The `SearchArgs` schema allows `genre` and a nonnegative `max_price`. The `RatingArgs` schema requires a positive integer `book_id`. `extra="forbid"` means unexpected fields are rejected.

**B. Register two functions as tools.** `@tool("search_books", args_schema=SearchArgs)` gives the existing search function a tool name, a description and a schema. The wrapper calls the **same ordinary Python function** you tested in Step 1.

**C. Publish a safe list.** The `TOOLS` dictionary is an **allowlist**. We publish descriptions in `SCHEMAS` so models know what they are permitted to request.

**D. Execute validated requests.** `execute_calls(...)` checks the requested name and arguments before running a tool. A request to `buy_book`, for example, is **blocked** because it is not in our allowlist.

**E. Interpret Qwen's response.** `parse_calls(...)` converts Qwen's observed XML-like tool-request text into Python dictionaries. `scalar(...)` converts text such as `"16"` into the number `16`. This parser is educational, not a universal parser for every model.

**F. Set rules for the assistant.** `SYSTEM` gives Qwen instructions about when to use tools, what information is missing from a catalog search and what not to invent.

In [ ]:
%%writefile demo_toolkit.py
"""LangChain wraps our two ordinary Python functions as schema-described tools."""
import json
from langchain_core.tools import tool
from pydantic import BaseModel, ConfigDict, Field
from demo_bookstore import search_books, get_book_rating

# Pydantic turns these fields into a machine-readable input schema.
class SearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    genre: str = Field(description="Book genre, for example sci-fi or fantasy")
    max_price: float = Field(ge=0, description="Maximum price in US dollars")

class RatingArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    book_id: int = Field(strict=True, gt=0, description="Positive catalog ID (not book title)")

# @tool attaches metadata and validation to our ordinary search function.
@tool("search_books", args_schema=SearchArgs)
def search_books_tool(genre: str, max_price: float) -> str:
    """Find matching books; returns IDs, titles and prices but NO ratings."""
    return json.dumps(search_books(genre, max_price))

@tool("get_book_rating", args_schema=RatingArgs)
def get_book_rating_tool(book_id: int) -> str:
    """Get the rating and number of reviews for a book's numeric ID."""
    return json.dumps(get_book_rating(book_id))

# The only tool functions we permit the models to request.
# Only these tools are permitted to be invoked.
TOOLS = {t.name: t for t in (search_books_tool, get_book_rating_tool)}
SCHEMAS = [
    {"type": "function", "function": {
        "name": t.name,
        "description": t.description,
        "parameters": t.args_schema.model_json_schema(),
    }} for t in TOOLS.values()
]

"""Validation boundary: proposed tool calls are NOT arbitrary Python execution."""
import json

# Validate requests in Python before running any function.
def execute_calls(calls):
    """Invoke only named, allowlisted, schema-validated read-only tools."""
    output = []
    for request in calls:
        name, args = request.get("name"), request.get("arguments")
        if name not in TOOLS:
            output.append({"name": name, "status": "blocked", "result": None})
            continue
        if not isinstance(args, dict):
            output.append({"name": name, "status": "invalid_arguments", "result": None})
            continue
        try:
            result = json.loads(TOOLS[name].invoke(args))
            output.append({"name": name, "status": "ok", "arguments": args, "result": result})
        except Exception as exc:
            output.append({"name": name, "status": "invalid_arguments", "error": str(exc)[:200], "result": None})
    return output

"""Teaching parser for Qwen3.5's observed XML-like tool requests.

Not a generic production parser: other models can use other protocols.
"""
import json
import re
BLOCK = re.compile(r"<tool_call(?:\s[^>]*)?>(.*?)</tool_call>", re.S | re.I)
FUNCTION = re.compile(r"<function=([^>]+)>(.*?)</function>", re.S | re.I)
PARAM = re.compile(r"<parameter=([^>]+)>(.*?)</parameter>", re.S | re.I)

def scalar(text):
    value = text.strip()
    try:
        return json.loads(value)  # numbers, true, false
    except json.JSONDecodeError:
        if value.lower() in ("true", "false"):
            return value.lower() == "true"  # model might say Python's True
        return value

# Parse a model response into structured requests; parsing is NOT execution.
def parse_calls(raw):
    calls = []
    for block in BLOCK.findall(raw):
        match = FUNCTION.search(block)
        if match:
            calls.append({"name": match.group(1).strip(),
                          "arguments": {key: scalar(val) for key, val in PARAM.findall(match.group(2))}})
            continue
        try:
            payload = json.loads(block)
        except (ValueError, TypeError):
            continue
        if isinstance(payload, dict):
            if isinstance(payload.get("function"), dict):
                payload = payload["function"]
            if isinstance(payload.get("name"), str):
                args = payload.get("arguments", {})
                if isinstance(args, str):
                    try: args = json.loads(args)
                    except ValueError: continue
                if isinstance(args, dict):
                    calls.append({"name": payload["name"], "arguments": args})
    return calls

def clean_answer(raw):
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    if "</think>" in raw:
        raw = raw.split("</think>")[-1]
    for ending in ("<|im_end|>", "<|endoftext|>", "<|eot_id|>"):
        raw = raw.replace(ending, "")
    return raw.strip()

SYSTEM = ("You help customers with this toy bookstore. You have two read-only tools. "
          "Catalog search returns NO ratings. For a request with a rating threshold, "
          "first search_books, then get_book_rating for the candidate's ID. "
          "Never invent catalog prices, ratings or book IDs. If no match, say so. "
          "When done, give a short answer with title, price and verified rating.")

if __name__ == "__main__":
    for schema in SCHEMAS:
        print("Tool available:", schema["function"]["name"])
        print("  schema:", json.dumps(schema["function"]["parameters"], indent=2))
    print("Manual tool call:", TOOLS["search_books"].invoke({"genre":"sci-fi","max_price":16}))
    print("Manual rating call:", TOOLS["get_book_rating"].invoke({"book_id":1}))
    assert parse_calls("<tool_call><function=search_books><parameter=genre>sci-fi</parameter><parameter=max_price>16</parameter></function></tool_call>")[0]["arguments"]["max_price"] == 16
    assert execute_calls([{"name":"buy_book","arguments":{}}])[0]["status"] == "blocked"
    print("Validation checks passed. No LLM was involved.")

### Tool metadata is not the tool result

Consider the tool declaration:

```python
@tool("get_book_rating", args_schema=RatingArgs)
def get_book_rating_tool(book_id: int) -> str:
    """Get the rating and number of reviews for a book's numeric ID."""
    return json.dumps(get_book_rating(book_id))
```

**What the model can see:** the tool's name, description, argument names, types, and constraints. **What the model cannot see until a tool runs:** the result stored in `RATINGS`.

The result is serialized as JSON so it can be carried between components. The function remains ordinary Python underneath the wrapper.

### Inspect the schemas and call the tools manually

The next cell is still **not using an LLM**. It will print the JSON schemas and invoke the registered tools through LangChain.

Look for `properties` and `required` in a schema. These tell the model which input fields exist and which are mandatory.

The sample calls use `.invoke({...})`, where the dictionary holds named arguments. The tool returns a JSON string, and `execute_calls(...)` parses that JSON back into Python data.

The script also checks that the unregistered name `buy_book` is blocked. A model's instructions must not bypass this Python-side restriction.

In [ ]:
!python demo_toolkit.py

### Checkpoint 2 — Function or tool?

Consider this request:

```json
{"name": "get_book_rating", "arguments": {"book_id": 1}}
```

This is **data representing a proposed function call**. It does not execute anything by itself. Explain the difference between (1) the request, (2) the validation step, and (3) the eventual Python function result.

Optional experiment: change `book_id` to `"Dune"` in a **copy** of a manual test. Why should the schema reject that value?

## Step 3 — Connect the tools to a local language model

Now we will use **Qwen3.5-4B**, running on the Colab GPU. The model sees a user request and tool descriptions, and may propose a tool call. **The model cannot directly run `search_books` or `get_book_rating`.** Our Python program parses and executes the proposed request.

### What is an agent loop?

An *agent loop* repeats the same cycle until there is a final answer (or a safety limit is reached):

`User question → Qwen requests a tool → Python validates and calls it → result is returned to Qwen → Qwen decides what to do next`

We implement this short loop directly in Python so that you can inspect it. **LangChain supplies the tool descriptions and schemas, not a prebuilt agent in this demo.** In the longer lab you will study the same idea with more tools and cases.

### Understand the Qwen script before loading the model

The next file, `demo_qwen.py`, first **loads the local model** and then defines a small **agent loop**.

**Model loading**

- `AutoProcessor` prepares messages for the Qwen model. Although Qwen3.5 is multimodal, our mini demo uses **text input only**.
- `AutoModelForMultimodalLM` loads the model itself.
- `BitsAndBytesConfig(load_in_4bit=True, ...)` requests **4-bit quantization**, lowering GPU memory usage compared with higher-precision weights.
- `model.eval()` selects evaluation/inference behavior rather than training.

**One agent turn**

1. `processor.apply_chat_template(...)` formats the conversation and tool descriptions in the model's expected format.
2. `model.generate(...)` produces the next model output; `max_new_tokens=320` caps its length.
3. `parse_calls(raw)` looks for proposed tool calls.
4. If the model requests a tool, `execute_calls(...)` invokes it in Python.
5. Tool results are appended to the conversation as messages with role `tool`, and the next model turn sees them.
6. If the model supplies a normal answer without a tool request, the loop finishes.

**Safety and control:** The loop is limited to four model turns, checks repeated requests and uses only the allowed tools. These safeguards are simple classroom examples, not complete production-grade protections.

In [ ]:
%%writefile demo_qwen.py
"""Use %run -i so model and processor remain in the notebook runtime."""
import os
from pathlib import Path
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab (Runtime -> Change runtime type).")
MODEL_ID = "Qwen/Qwen3.5-4B"
if "model" in globals() and "processor" in globals():
    print("Reusing the already loaded Qwen model.")
else:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True)
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, quantization_config=quant, device_map="auto",
        dtype=torch.float16, low_cpu_mem_usage=True)
    model.eval()
tokenizer = processor.tokenizer
print("Local model:", MODEL_ID)
print("Quantized (4-bit):", bool(getattr(model, "is_loaded_in_4bit", False)))
print("Model ready. First download can take time.")

"""A transparent *custom* agent loop using LangChain's tool definitions.

LangChain provides @tool and schemas, but this small loop is OUR Python code:
model -> validate/execute tool -> append result -> repeat -> answer.
"""
import json, time
import torch
from demo_toolkit import SCHEMAS, parse_calls, clean_answer, execute_calls, SYSTEM


# One inference turn: format messages (and tools), generate tokens, decode text.
def qwen_reply(history, model, processor, tokenizer):
    inputs = processor.apply_chat_template(
        history, tools=SCHEMAS, enable_thinking=False,
        add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=320,
                                do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new = output[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(new, skip_special_tokens=False).strip()

# Our custom orchestration loop: model -> tool -> result -> next model turn.
def run_local(question, model, processor, tokenizer, verbose=True):
    history = [{"role": "system", "content": SYSTEM},
               {"role": "user", "content": question}]
    trace = {"model": "Qwen/Qwen3.5-4B", "question": question,
             "status": "running", "answer": None, "calls": [], "turns": []}
    seen = None
    start = time.perf_counter()
    for step in range(1, 5):  # bounded: at most 4 model turns
        raw = qwen_reply(history, model, processor, tokenizer)
        calls = parse_calls(raw)
        if verbose: print(f"\nMODEL TURN {step}:", clean_answer(raw) if not calls else raw[:800])
        trace["turns"].append({"turn": step, "raw": raw, "requested": calls})
        if not calls:
            if "<tool_call" in raw:  # malformed tool markup
                trace["status"] = "invalid_tool_call"
            else:
                trace["answer"] = clean_answer(raw)
                trace["status"] = "final_answer" if trace["answer"] else "empty_answer"
            break
        signature = json.dumps(calls, sort_keys=True)
        if signature == seen or len(trace["calls"]) + len(calls) > 4:
            trace["status"] = "repeated_or_excessive_tool_calls"
            break
        seen = signature
        # Python, not Qwen, performs the real tool execution.
        results = execute_calls(calls)
        trace["calls"].extend(results)
        if verbose:
            for r in results: print(" PYTHON executes:", r["name"], "→", json.dumps(r.get("result")))
        # Record the model request so the next turn knows which tool was called.
        history.append({"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
            for c in calls]})
        for c, r in zip(calls, results):
            history.append({"role": "tool", "name": c["name"],
                            "content": json.dumps(r)})
    else:
        trace["status"] = "max_turns"
    trace["seconds"] = round(time.perf_counter() - start, 2)
    if verbose:
        print("FINAL:", trace["answer"] or "[no completed answer]")
        print("STATUS:", trace["status"], "TOOL CALLS:", len(trace["calls"]))
    return trace

DEMO_QUESTION = globals().get("DEMO_QUESTION", "Find a sci-fi book under $16 with a rating above 4.5.")
print("USER:", DEMO_QUESTION)
LOCAL_TRACE = run_local(DEMO_QUESTION, model, processor, tokenizer)

### Why the loop stores conversation history

The variable `history` starts with a **system instruction** and the **user question**. After the first tool call, it gains (1) a message recording the assistant's tool request and (2) a message recording Python's tool result.

When the next model turn begins, Qwen receives the updated history. That is how it can learn that `search_books` found book ID `1` and then request its rating.

Without returning the tool result, the model would not have verified data for the next step. This is the key difference between a **single tool call** and a **multi-step agent workflow**.

### Run the local Qwen agent

Set `DEMO_QUESTION` and execute the script with `%run -i`. The `-i` flag is important because it makes the loaded `model` and the resulting `LOCAL_TRACE` available in the notebook session.

**Prediction:** How many model turns and tool calls might be needed? The question contains both a **catalog constraint** (genre and price) and a **rating constraint**.

The first download and load can take longer than later runs. In a correct execution, watch for a tool request to `search_books` followed by `get_book_rating`. The actual wording and number of turns can vary: models are not guaranteed to produce identical output on every run.

In [ ]:
DEMO_QUESTION = "Find a sci-fi book under $16 with a rating above 4.5."
%run -i demo_qwen.py

### Interpreting the local trace

Read the output in order:

- **`MODEL TURN`** means Qwen generated a new response. One turn may request tools; another may finish with ordinary text.
- **`PYTHON executes:`** means the program really invoked the selected tool and received a result.
- **`FINAL:`** is the assistant's final natural-language response.
- **`STATUS:`** reports whether the loop finished normally or hit a guard such as the maximum turn count.
- **`TOOL CALLS:`** counts executed requests, not model turns.

A typical successful route is `search_books → get_book_rating → final answer`, with two tool calls across three model turns. That sequence is a teaching expectation, **not an unconditional promise**.

**Checkpoint 3:** Find the exact value that told Qwen which `book_id` to pass to `get_book_rating`. Was it invented by the model, or returned by `search_books`?

## Step 4 — Compare with a model hosted on Groq

The local and hosted versions share exactly the same bookstore and tool functions. The difference is **where the model runs**:

| Local Qwen | Hosted model via Groq |
|---|---|
| Runs on your Colab GPU | Runs on Groq's servers |
| Transformers loads model weights | Groq API sends requests and returns responses |
| Model may emit tool requests in Qwen's XML-like format | Groq returns structured tool calls with IDs |
| Your Colab Python program executes tools | Your Colab Python program **still** executes tools |

**Before you run the next cell:** Store your API key in **Colab → Secrets** as `GROQ_API_KEY` and allow notebook access. The code reads the secret without printing it. You can skip this step if you do not have Groq access and still complete the local demonstration.

We start with `qwen/qwen3.8-27b`. You can optionally try `openai/gpt-oss-20b` later; model availability and rate limits may change.

**Check your understanding:** When Groq requests `get_book_rating`, does Groq execute that local Python function, or does your Colab runtime?

### Read the Groq script

`demo_groq.py` does two main things:

1. **Connect safely.** The script reads `GROQ_API_KEY` from Colab Secrets (or an existing environment variable), and creates a Groq client. The key should never be printed or included in submitted notebooks.
2. **Run the same tool loop remotely.** `client.chat.completions.create(...)` sends the user question, available tool schemas, and conversation history to a hosted model. When Groq returns tool-call requests, the **same local `execute_calls(...)` function** runs the bookstore functions.

You will see `tool_call_id` in the code. Each returned tool result is associated with the exact tool request that created it. This is how the hosted chat API keeps a multi-step conversation consistent.

Even though the model is hosted elsewhere, **your actual bookstore data and Python functions remain in Colab**. The request and selected tool arguments are sent to Groq, not an executable Python function.

In [ ]:
%%writefile demo_groq.py
"""Read Groq credentials from Colab Secrets. Never print/store the key in code."""
import os
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key: os.environ["GROQ_API_KEY"] = key
    except Exception:
        pass
GROQ_CLIENT = None
if os.environ.get("GROQ_API_KEY"):
    from groq import Groq
    GROQ_CLIENT = Groq(api_key=os.environ["GROQ_API_KEY"])
    print("Groq client ready. Key not shown.")
else:
    print("No Groq key. Save GROQ_API_KEY in Colab Secrets and grant access.")

"""Same schemas and Python tools, but a hosted Groq model chooses calls."""
import json, time
from demo_toolkit import SCHEMAS, execute_calls, SYSTEM

# Groq hosts the model; tools and validation still run in Colab.
def run_groq(client, model_id, question, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    trace = {"model": model_id, "question": question, "status": "running",
             "answer": None, "calls": [], "turns": []}
    seen = None
    start = time.perf_counter()
    try:
        for step in range(1, 5):
            # Send conversation history and tool definitions to the hosted model.
            response = client.chat.completions.create(
                model=model_id, messages=messages, tools=SCHEMAS,
                tool_choice="auto", parallel_tool_calls=False,
                max_completion_tokens=320)
            message = response.choices[0].message
            requested = message.tool_calls or []
            trace["turns"].append({"turn": step, "requested": [c.function.name for c in requested]})
            if verbose: print(f"\nGROQ TURN {step}:", [c.function.name for c in requested] or "final text")
            if not requested:
                trace["answer"] = (message.content or "").strip()
                trace["status"] = "final_answer" if trace["answer"] else "empty_answer"
                break
            parsed = []
            for call in requested:
                try: args = json.loads(call.function.arguments)
                except (ValueError, TypeError): args = None
                parsed.append({"name": call.function.name, "arguments": args})
            signature = json.dumps(parsed, sort_keys=True)
            if signature == seen or len(trace["calls"]) + len(parsed) > 4:
                trace["status"] = "repeated_or_excessive_tool_calls"
                break
            seen = signature
            results = execute_calls(parsed)
            trace["calls"].extend(results)
            if verbose:
                for r in results: print(" PYTHON executes:", r["name"], "→", json.dumps(r.get("result")))
            # Tool call IDs must match the corresponding tool-result messages.
            # Preserve the returned call IDs to match later tool-result messages.
            messages.append({"role": "assistant", "content": message.content,
                             "tool_calls": [{"id": c.id, "type": "function", "function": {
                                 "name": c.function.name, "arguments": c.function.arguments}}
                                 for c in requested]})
            for c, r in zip(requested, results):
                messages.append({"role": "tool", "tool_call_id": c.id,
                                 "content": json.dumps(r)})
        else:
            trace["status"] = "max_turns"
    except Exception as exc:
        trace["status"] = "api_error"
        trace["error"] = f"{type(exc).__name__}: {exc}"[:400]
    trace["seconds"] = round(time.perf_counter() - start, 2)
    if verbose:
        print("FINAL:", trace["answer"] or "[no completed answer]")
        print("STATUS:", trace["status"], "TOOL CALLS:", len(trace["calls"]))
        if trace["status"] == "api_error": print("ERROR:", trace["error"])
    return trace

GROQ_MODEL = globals().get("GROQ_MODEL", "qwen/qwen3.8-27b")
DEMO_QUESTION = globals().get("DEMO_QUESTION", "Find a sci-fi book under $16 with a rating above 4.5.")
if GROQ_CLIENT is not None:
    print("USER:", DEMO_QUESTION)
    GROQ_TRACE = run_groq(GROQ_CLIENT, GROQ_MODEL, DEMO_QUESTION)
else:
    print("Groq test skipped: configure key in Colab Secrets and rerun this cell.")

### Hosted tool calling still needs local execution

The hosted API typically supplies a structured request resembling:

```json
{"name": "get_book_rating", "arguments": "{\"book_id\": 1}"}
```

Groq does not automatically run our local `get_book_rating` function. Python must extract and validate the arguments, call the function, and send the result back to the hosted model with the matching request ID. That is what our `run_groq(...)` function demonstrates.

### Execute the Groq comparison

The default model is `qwen/qwen3.8-27b`. You can change `GROQ_MODEL` to `"openai/gpt-oss-20b"` for an optional second test.

If the notebook reports that the Groq key is missing, return to **Colab → Secrets**, add `GROQ_API_KEY`, allow this notebook to read it, then rerun the cell. Requests may also fail due to model availability or rate limits; those errors should be reported rather than treated as model answers.

**Prediction:** Will this model necessarily choose exactly the same tool-call order as local Qwen? Why or why not?

In [ ]:
GROQ_MODEL = "qwen/qwen3.8-27b"
%run -i demo_groq.py

### Compare the two traces

Compare the outputs from `LOCAL_TRACE` and `GROQ_TRACE` before computing evaluation scores.

| Feature | What to find |
|---|---|
| First tool call | Was `search_books` requested? |
| Second tool call | Was `get_book_rating` requested after obtaining an ID? |
| Tool results | Did both tools return the expected facts? |
| Final answer | Is the answer consistent with the tool results? |
| Duration | Which run took longer in **this particular** experiment? |

Different responses do not necessarily mean one model is wrong. What matters is whether the final answer satisfies the question and is supported by real tool outputs.

## Step 5 — Evaluate what actually happened

A successful-looking answer is not enough. We want to check whether the agent **used the appropriate tools** and whether its **facts are supported by the returned results**.

Our short evaluation is a checklist, not a scientific benchmark. It checks:

- **Completed answer:** Did the agent finish with a final response?
- **Tool order:** Was `search_books` called before `get_book_rating`?
- **Verified results:** Did the tools actually return the expected price and rating?
- **Answer facts:** Does the answer mention *Dune*, `$14.99`, and `4.67`?
- **Runtime:** How long did this single execution take?

Do not treat one run as evidence that a model is generally more accurate or faster. Larger evaluations require multiple questions, repeated runs, failure cases, and comparable conditions.

### How the evaluation script works

`demo_evaluate.py` computes the **expected answer from the toy bookstore functions**, not from a model's memory. It then examines each saved trace.

- `grade(trace)` accepts one execution trace and returns a dictionary of results.
- `tools_in_order` checks whether a successful search happened before a successful rating lookup.
- `verified_tool_facts` checks whether the required book and rating were actually returned.
- `answer_mentions_facts` checks three strings in the final text. It is a deliberately **limited** test: the answer could still have unsupported extra claims.
- `seconds` records execution time; one timing sample is not a robust speed comparison.

A row with `False` does not always tell you the exact cause. Inspect the original trace to determine whether a tool failed, the model skipped a step, or the final response omitted a fact.

In [ ]:
%%writefile demo_evaluate.py
"""Small transparent checklist, NOT a statistically meaningful benchmark."""
from demo_bookstore import search_books, get_book_rating

EXPECTED_BOOK = search_books("sci-fi", 16)[0]
EXPECTED_RATING = get_book_rating(EXPECTED_BOOK["id"])

# Check transparent observable facts in one recorded agent execution.
def grade(trace):
    if not trace:
        return {"model": "not run", "task_done": False, "tools_in_order": False,
                "verified_tool_facts": False, "answer_mentions_facts": False,
                "seconds": None}
    calls = trace.get("calls", [])
    names = [item.get("name") for item in calls if item.get("status") == "ok"]
    ordered = ("search_books" in names and "get_book_rating" in names
               and names.index("search_books") < names.index("get_book_rating"))
    book_in_results = any(item.get("name") == "search_books" and item.get("status") == "ok"
                          and isinstance(item.get("result"), list)
                          and any(b.get("id") == 1 and b.get("price") == 14.99
                                  for b in item["result"]) for item in calls)
    rating_in_results = any(item.get("name") == "get_book_rating" and item.get("status") == "ok"
                            and isinstance(item.get("result"), dict)
                            and item["result"].get("book_id") == 1
                            and item["result"].get("average_rating") == 4.67
                            for item in calls)
    answer = trace.get("answer") or ""
    return {"model": trace.get("model"),
            "task_done": trace.get("status") == "final_answer",
            "tools_in_order": ordered,
            "verified_tool_facts": book_in_results and rating_in_results,
            "answer_mentions_facts": all(word in answer.lower()
                                         for word in ("dune", "14.99", "4.67")),
            "seconds": trace.get("seconds")}

if "LOCAL_TRACE" not in globals() or "GROQ_TRACE" not in globals():
    print("Run both models first for a fair side-by-side check.")
rows = [grade(globals().get("LOCAL_TRACE")), grade(globals().get("GROQ_TRACE"))]
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(rows))
except ImportError:
    for item in rows: print(item)
print("These checks only verify selected observable facts (not full factual correctness).")
print("One run is a classroom demonstration, not a statistical model ranking.")

### Run the evaluation

The output table contains one row for local Qwen and one for Groq (or a `not run` row if a model was skipped). Carefully distinguish:

**Observed success:** The model's output passed our defined checks.

**General reliability:** How often the model would pass across many different questions, repeats, and failure scenarios. Our short demonstration does **not** measure that.

Try to locate at least one fact in the final answer and trace it backward to a concrete Python tool result.

In [ ]:
%run -i demo_evaluate.py

### Checkpoint 4 — Evaluate the evaluation

Before the closing reflection, consider:

1. Could a model mention “Dune”, `$14.99`, and `4.67` **without having called either tool**? Which other checks would catch this?
2. Why would a test containing many differently worded questions be stronger than testing one question once?
3. If a model selects the wrong tool but gives a plausible final answer, should we consider that run fully successful? Explain your criterion.

## Reflection and next steps

Write short answers in your notes. Refer to the code and the model traces, not only to a final recommendation.

1. What is the difference between an **ordinary Python function**, a **registered tool**, and an **agent loop**?
2. Who chooses a tool, and who **executes** it?
3. Why does the search result not contain ratings? What extra information is needed to check a rating threshold?
4. What is an **argument schema**, and what happens if a model requests a tool that isn't allowed?
5. How does the local Qwen workflow differ from the Groq workflow? What remains unchanged?
6. What does our evaluation **verify**, and what can it **not verify**?
